# 05 - Conclusiones

Las conclusiones se apoyan en la inspección, limpieza, EDA y PCA. Se separa evidencia observada de interpretación.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
ROOT = Path("..").resolve()

df = pd.read_csv(ROOT / "data" / "processed" / "streaming_users_processed.csv")
log = pd.read_csv(ROOT / "logs" / "pipeline_log.csv")
log


,Paso,Descripción,Filas,Nulos,Retención (%)
0,0,Carga del dataset original en una copia de tra...,8160,753,100.00
1,1,Eliminación de duplicados exactos sin modifica...,8034,753,98.46
2,2,Resolución de user_id repetidos priorizando fe...,8000,743,98.04
3,3,"Estandarización de categorías en plan, país y ...",8000,743,98.04
4,4,Conversión de valores imposibles a nulos: edad...,8000,1019,98.04
5,5,Imputación justificada con medianas/modas segm...,8000,0,98.04
6,6,Winsorización superior: monthly_watch_time_min...,8000,0,98.04
7,7,Normalización final de tipos y exportación del...,8000,0,98.04


In [2]:
resumen = {
    "usuarios_finales": len(df),
    "retencion_final": log.iloc[-1]["Retención (%)"],
    "consumo_mediano": df["monthly_watch_time_mins"].median(),
    "edad_mediana": df["age"].median(),
    "tickets_promedio": round(df["customer_support_tickets"].mean(), 2)
}
resumen


{'usuarios_finales': 8000,
 'retencion_final': np.float64(98.04),
 'consumo_mediano': np.float64(772.45),
 'edad_mediana': np.float64(33.0),
 'tickets_promedio': np.float64(0.83)}

## Hallazgos principales

- La base necesitaba limpieza antes del analisis: habia duplicados, categorias inconsistentes, fechas invalidas, valores imposibles y extremos.
- La preparacion permitio llegar a una base final sin nulos, sin duplicados y con las mismas columnas originales.
- El consumo mensual es la variable mas intuitiva para empezar a describir perfiles de usuario.
- La edad aporta contexto, pero no explica por si sola el comportamiento de visualizacion.
- Al sumar plan, tickets de soporte y genero favorito, el analisis permite observar perfiles mas completos: intensidad de uso, posible friccion operativa y preferencias de contenido.

## Interpretacion general

El valor del trabajo no esta solamente en limpiar la base, sino en poder explicar por que se limpio de esa manera. Por ejemplo, un `user_id` repetido no se resolvio al azar: se priorizo una fecha real, un consumo mensual plausible, cercania al consumo tipico y completitud.

La winsorizacion tampoco se aplico por moda. Se uso porque valores extremos como consumos o tickets exageradamente altos podian distorsionar graficos, medidas resumen y PCA. En palabras simples: se conservaron los usuarios, pero se evito que valores poco realistas manejaran toda la lectura.

## PCA

PCA se aplico sobre `age`, `monthly_watch_time_mins` y `customer_support_tickets` despues de estandarizar. Esto fue necesario porque las variables tienen escalas distintas: anios, minutos y cantidad de tickets.

La lectura de PCA debe ser prudente. Sirve para resumir perfiles numericos y observar si existen agrupamientos, pero no reemplaza el analisis exploratorio ni permite afirmar causalidad.

## Limitaciones

- El dataset no incluye churn, antiguedad, precio pagado, satisfaccion, dispositivo ni historial detallado de sesiones.
- Las conclusiones son descriptivas: explican patrones observados, no causas definitivas.
- La imputacion y la winsorizacion son decisiones justificadas, pero siguen siendo decisiones analiticas que deben declararse.
- La fecha de ultimo login permite una lectura parcial de actividad, pero no alcanza para medir retencion real.

## Proximos pasos

- Incorporar variables comerciales como precio, promociones, fecha de alta y cancelaciones.
- Agregar metricas temporales: cantidad de sesiones, dias activos y evolucion mensual del consumo.
- Analizar retencion o riesgo de baja si se incorpora una variable objetivo.
- Validar las reglas de limpieza con una mirada de negocio antes de usar el proceso en produccion.

## Conclusion final

El proyecto deja una base ordenada, trazable y lista para analisis. La evidencia muestra que el comportamiento de usuarios de streaming no puede resumirse con una sola variable: consumo, soporte, plan, edad y genero favorito aportan piezas distintas del perfil.

La conclusion mas importante es metodologica: antes de interpretar, hay que preparar bien los datos. Sin esa etapa, el analisis puede sonar convincente, pero apoyarse en errores de carga, duplicados o extremos poco realistas.